# Reentrenar el latín (adaptador v2)

El adaptador actual (**v1**) se entrenó con 1,500 frases cortas (~10 palabras) y falla con frases complejas: subjuntivo, condicionales, subordinadas. El **v2** parte del v1 y sigue entrenando **solo con ~1,500 frases largas (8-22 palabras) que el v1 nunca vio**. Se guarda en `adaptador_latin_v2/`; el v1 no se toca.

Al final compara **v1 vs v2 vs sin adaptador** con 10 frases complejas (`frases_complejas.txt`) y **descarga un zip** con el adaptador v2 y los resultados.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`. Después `Entorno de ejecución → Ejecutar todas` y deja la pestaña abierta.

| Paso | Tiempo aprox. en T4 |
|---|---|
| Instalar y descargar modelos | 5-10 min |
| Entrenar v2 | 70-90 min |
| Comparar v1 / v2 / sin adaptador (10 frases) | ~20 min |

**Si Colab se desconecta:** con la casilla de Drive marcada (sección 3) el avance se guarda en `Mi unidad/traductor_p1/`; vuelve a abrir este notebook, `Ejecutar todas`, y el entrenamiento sigue desde el último guardado (cada 200 pasos).

## 1. Verificar GPU

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

## 2. Obtener el código e instalar librerías

In [ ]:
REPO = "AndreaHortaHernandez/Traductormultilingue"

import os
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -U transformers sentence-transformers sacrebleu peft
# Colab trae torchao y peft (LoRA) se niega a cargar con esa versión; no se usa.
!pip uninstall -q -y torchao

## 3. Respaldo en Google Drive (recomendado)

Guarda `adaptador_latin_v2/` y `salidas/` en `Mi unidad/traductor_p1/` mientras entrena. Te pide permiso para tu Drive.

In [ ]:
RESPALDO_EN_DRIVE = True  #@param {type:"boolean"}

import os, shutil
if RESPALDO_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/traductor_p1"
    for carpeta in ("adaptador_latin_v2", "salidas"):
        destino = f"{base}/{carpeta}"
        if os.path.islink(carpeta):
            continue
        if os.path.isdir(carpeta):
            shutil.copytree(carpeta, destino, dirs_exist_ok=True)
            shutil.rmtree(carpeta)
        os.makedirs(destino, exist_ok=True)
        os.symlink(destino, carpeta)
    print("Respaldando en", base)
else:
    print("Sin respaldo: si Colab se desconecta, el entrenamiento empieza de nuevo.")

## 4. Entrenar el v2

Imprime la pérdida cada 20 pasos (debe bajar poco a poco; empieza más baja que la del v1 porque parte de él) y cuánto falta.

In [ ]:
import gc, torch, afinar_latin
afinar_latin.afinar_v2()
gc.collect(); torch.cuda.empty_cache()  # libera la GPU para la siguiente celda

## 5. Cargar el traductor con v1 y v2

In [ ]:
from modelos import Traductor, Verificador
traductor = Traductor()  # con el v1
import config
traductor.agregar_adaptador("v2", config.RUTA_ADAPTADOR_LATIN_V2)
verificador = Verificador()

## 6. Comparar v1 / v2 / sin adaptador con frases complejas

Misma cadena, mismas frases; solo cambia el adaptador del latín. Imprime el resumen, la similitud final por frase y cómo regresa cada una al español.

In [ ]:
limite = 10  #@param {type:"integer"}

import comparar
comparacion = comparar.comparar(
    traductor, verificador, ruta_frases="frases_complejas.txt",
    limite=limite, variantes=comparar.VARIANTES_V2,
)

## 7. Descargar el adaptador v2 y los resultados

Baja `v2_y_resultados.zip` (~40 MB) a tu compu. Pásaselo a quien mantiene el repo para subirlo.

In [ ]:
%cd /content/repo
!zip -qr v2_y_resultados.zip adaptador_latin_v2 salidas
from google.colab import files
files.download("/content/repo/v2_y_resultados.zip")

## 8. (Opcional) Probar la cadena con el v2

Escribe una frase y compara cómo queda con el v1 y con el v2.

In [ ]:
frase = "Quisiera comer algo diferente esta noche, pero no sé qué."  #@param {type:"string"}

from cadena import traducir_en_cadena, imprimir_pasos
for nombre, adaptador in (("v1", True), ("v2", "v2")):
    print(f"
==================== {nombre} ====================")
    t = traductor if adaptador is True else comparar._ConAdaptador(traductor, adaptador)
    imprimir_pasos(frase, traducir_en_cadena(frase, config.CADENA_POR_DEFECTO, t, verificador))